# PPMI_PrimeKGL
Copied from the original project. Paths are relative to the main folder.
Full PrimeKG embeddings and entity indices are shared by all four cohorts.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'PPMI'
METHOD = 'PrimeKGL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
# Full PrimeKG is used directly: no seed extraction, blacklist, or Top-k.
RAW_KG_PATH = str(raw_primekg_path())
SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
if not Path(RAW_KG_PATH).is_file():
    raise FileNotFoundError(RAW_KG_PATH)
print("Full PrimeKG source:", RAW_KG_PATH)
print("All cohorts share graph-trained assets in:", SHARED_PRIMEKG)


In [ ]:
from pathlib import Path
import sys
_split_root = CLINICALKG_ROOT
if str(_split_root) not in sys.path:
    sys.path.insert(0, str(_split_root))
from experiments.experiment_split import split_cohort_arrays, cohort_indices
# Fixed patient split: about 64% train, 16% validation, 20% test.
# Keep this split unchanged across model variants and all five training runs.

import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings('ignore')

# ================= ⚡ 配置区 =================
PPMI_DATA_PATH = str(DATASET_DIR / 'PPMI诊断.csv')
SEED_CSV_PATH = str(WORK_DIR / 'PPMI_Seed_DementiaHKG.csv')
REFINED_KG_PATH = str(raw_primekg_path())

OUTPUT_GRAPH_PATH = str(SHARED_PRIMEKG / 'PrimeKG-G.csv')
OUTPUT_DICT_PATH = str(WORK_DIR / 'PPMI-PrimeKG-Entities.json')
OUTPUT_SPLIT_PATH = str(WORK_DIR / 'PPMI_Dataset_Split.csv')

MAX_SEQ_LEN = 35
PAD_TOKEN = "[PAD]"

def build_patient_mapping_ppmi():
    print("==================================================")
    print("🔗 阶段 3: 构建患者动态映射与客观图谱格式化 (已省略常模计算)...")

    # 1. 提取图谱结构
    try:
        kg_df = pd.read_csv(REFINED_KG_PATH)
        triplets_df = kg_df[['x_name', 'relation', 'y_name']].copy()
        triplets_df.columns = ['head', 'relation', 'tail']
        
        triplets_df.to_csv(OUTPUT_GRAPH_PATH, index=False, encoding='utf-8')
        print(f"   ✅ 标准化图谱已生成: {OUTPUT_GRAPH_PATH} (共 {len(triplets_df)} 条边)")
        print("   🔒 [安全检查]: 该图谱中不存在任何患者节点，已彻底阻断 Transductive 泄露。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {REFINED_KG_PATH}，请确保上一阶段已成功运行。")
        return

    # 2. 加载映射字典
    seed_df = pd.read_csv(SEED_CSV_PATH)
    # PPMI 的特征映射类别在 Step2 中定义为 Phenotype
    mapping_df = seed_df[seed_df['Category'] == 'Phenotype'] 
    ppmi_to_pkg_map = dict(zip(mapping_df['PPMI_Feature'], mapping_df['PrimeKG_Node']))
    print(f"\n   📊 成功加载 {len(ppmi_to_pkg_map)} 条临床特征映射规则。")

    # 3. 加载完整数据集
    try:
        ppmi_df = pd.read_csv(PPMI_DATA_PATH, low_memory=False)
    except FileNotFoundError:
        print(f"   ❌ 找不到患者数据 {PPMI_DATA_PATH}。")
        return

    print("\n   🔀 正在执行安全的随机数据集划分 (Train: 70%, Val: 10%, Test: 20%)...")
    np.random.seed(42) 
    
    # 获取所有有效患者 ID，兼容 Subject 或 PATNO
    def get_pid(row):
        if 'Subject' in row and pd.notna(row['Subject']): return str(row['Subject']).strip()
        if 'PATNO' in row and pd.notna(row['PATNO']): return str(row['PATNO']).strip()
        return None

    ppmi_df['patient_id_temp'] = ppmi_df.apply(get_pid, axis=1)
    # Reuse the fixed patient split used by every downstream model.
    from experiments.experiment_split import export_manifest, training_rows
    export_manifest("PPMI", OUTPUT_SPLIT_PATH)

    # 省略常模计算，使用直接的异常判定逻辑
    def check_abnormal(val):
        return val > 0

    patient_entities = {}
    valid_mapped_patients = 0
    total_features_mapped = 0

    print(f"\n   ⏳ 正在为所有患者建立图谱实体序列...")
    
    for index, row in ppmi_df.iterrows():
        patient_id = row.get('patient_id_temp')
        if not patient_id: continue

        active_nodes = []
        for col_name, val in row.items():
            if col_name in ppmi_to_pkg_map:
                val_str = str(val).strip().lower()
                if pd.notna(val) and val_str not in ['', 'nan', '-4', '-1']:
                    try:
                        num_val = float(val)
                        if check_abnormal(num_val):
                            primekg_node = ppmi_to_pkg_map[col_name]
                            active_nodes.append(primekg_node)
                            total_features_mapped += 1
                    except ValueError:
                        pass

        active_nodes = list(set(active_nodes))
        seq_len = len(active_nodes)
        
        if seq_len > MAX_SEQ_LEN:
            active_nodes = active_nodes[:MAX_SEQ_LEN]
            attention_mask = [1] * MAX_SEQ_LEN
        else:
            pad_length = MAX_SEQ_LEN - seq_len
            attention_mask = [1] * seq_len + [0] * pad_length
            active_nodes = active_nodes + [PAD_TOKEN] * pad_length

        patient_entities[patient_id] = {
            "input_nodes": active_nodes,
            "attention_mask": attention_mask
        }
        
        if seq_len > 0:
            valid_mapped_patients += 1

    with open(OUTPUT_DICT_PATH, 'w', encoding='utf-8') as f:
        json.dump(patient_entities, f, indent=4, ensure_ascii=False)

    print("\n==================================================")
    print("💾 阶段 3 任务完成！")
    print(f"   👨‍⚕️ 处理患者总数: {len(ppmi_df)}")
    print(f"   🔗 成功映射患者数: {valid_mapped_patients}")
    print(f"   📈 映射特征总次: {total_features_mapped}")
    print(f"   📄 图谱序列已保存至: {OUTPUT_DICT_PATH}")
    print("==================================================")

if __name__ == "__main__":
    build_patient_mapping_ppmi()

In [ ]:
# Same DistMult score/loss; one shared full-graph pretraining job.
from clinicalKG.full_primekg import train_full_primekg, plot_loss
RESUME = False  # Set True to continue PrimeKG-DistMult-last.pt.
NUM_EPOCHS = 50  # Total target epochs, shared across cohorts.
EMBED_DIM = 128
BATCH_SIZE = 512
LR = 0.005
training_state = train_full_primekg(epochs=NUM_EPOCHS, embed_dim=EMBED_DIM,
                                   batch_size=BATCH_SIZE, lr=LR, resume=RESUME)
plot_loss()
